In [27]:
import polars as pl
import sys
import os
sys.path.append(fr'C:\Users\{os.getlogin()}\88c6afda0b696ca552ccd7000b7ae067\RFS-MAP\SAS-DATA-PULL')
from Agora import C1BConnections
import pandas as pd
import numpy as np

In [28]:
cnx=C1BConnections()

Ready to open connections.


In [29]:
cnx.connect_to_SAS()

SAS Connection established. Subprocess id is 4696



 Connected to SAS

In [ ]:
cnx.SAS_Connection.submit(
'''
libname trm '/sasdata/area51/group/RISK_MODELING/TRM/Analysis2026Q2';

%let folder=/sasdata/windows/fnbmcorp/Risk/Modeling/Model Development/Vendor Engagements/Active/01 - Adhoc/2026Q2_RR_Anlaysis/FilterCRS;


/**********************************Macro 1.1 - Step1: Read Mailfile Macro*********************************/
%macro readmailfile_trm(startdate,ds); 

%let enddate = %sysfunc(intnx(month,"&startdate."d,0,E),date9.); 
%put &enddate.;
/*Read the mailfile*/
data &ds._mailfiles; 
	set datalake.experianprescreen (keep= 
CAMPAIGN_CODE
CAMPAIGN_DATE 
CAMPFLAG1
CAMPFLAG2 
CRS18_P
CRS18_SEGMENT
ENCRYPTED_PIN
NRM16 NRM16_TIER NRM16_TIER_VS4 
RISK_SCORE
RISK_SCORE_TYPE
RESERVATION_SEQUENCE 
OFFER_CODE_FIRST_7 
OFFER_CODE_LAST_3  
response_model_type 
TRM10_Score TRM10_TIER prospect_type 
SCORE_VANTAGE_VALUE_V3 SCORE_VANTAGE_VALUE_V3_SEGMENT TM12_3BR  TIMES_MAILED_12MO_CNT SURNAME FIRST_NAME STREET_NUMBER DOB endmark);

where  "&startdate."d<=CAMPAIGN_DATE<="&enddate."d and 
CAMPFLAG1  in ('V','A','D','M','N','R','W','C','B','G','I','J','Q')
and endmark not like '%S%';

vantage3 = score_vantage_value_v3*1;

TRM_Score = TRM10_Score*1;

ReservationNumber = cat(OFFER_CODE_FIRST_7,RESERVATION_SEQUENCE);

/*IF (550 <= VANTAGE3 <=600 AND RISK_SCORE>570) OR (601 <= VANTAGE3 <= 639 AND RISK_SCORE>559);*/

TM12_3RR = TM12_3BR *1;

if campflag1 in ('V','A','D','M','N','R','W','C') then rm_flag=0;

if campflag1 in ('B','G','I','J','Q') then rm_flag=1;

run;


%mend readmailfile_trm;



/**********************************Macro 1.2 - Step2: get response from caps data base********************/
%macro getresponse_trm(startdate,ds); 

/*initiate date macro*/
%let minappdate = %sysfunc(intnx(month,"&startdate."d,-1,B),date9.); 
%let maxappdate = %sysfunc(intnx(month,"&startdate."d,3,E),date9.);
%let campstartdate = %sysfunc(intnx(month,"&startdate."d,-1,B),date9.); 
%let pqappdate = %sysfunc(intnx(month,"&startdate."d,-2,B),date9.); 

%put &minappdate.;
%put &maxappdate.;
%put &campstartdate.;
%put &pqappdate.;

/*delete accounts that are fraud*/
%let frauddecs = 
'Declined: Identity Verification', 'Declined: OFAD'
,'Declined: Precise ID KIQ Grading Decision Code Check'
,'Failed Fraud Check SSN Validation'
,'Failed Fraud Match Letter Sent'
,'Failed IP Fraud Score Check'
,'Failed Precise ID Decision Code Check'
,'Failed Precise ID KIQ Grading Decision Code Check'
,'Failed Red Flag - Existing App/Acct Match'
,'PQ Failed IP Fraud Check'
,'PQ Failed IP Velocity'
,'Precise ID Information: Failed to Complete Information Retrieval'
,'Device ID Decline'
,'PQ Device Id Cancel'
,'PQ Device Id Decline'
,'2nd device ID Decline'
,'Precise ID Device Evaluation Information: Failed to Complete Information Retrieval'
,'Failed Precise ID Device Evaluation Decision Code Check'
,'Failed Precise ID Device Evaluation KIQ Grading Decision Code Check'
,'Skip PQ Device Id Cancel'
,'Skip Prequal Failed IP Fraud Score Check'
,'Skip PQ Failed IP Velocity'
,'Declined: Potential Identity Theft'
,'CrossCore Failed Fraud Shield 5'
,'CrossCore Failed Fraud Shield 13'
,'CrossCore Failed Fraud Shield 14'
,'CrossCore Failed Fraud Shield 25'
,'CrossCore Failed Exception Score 9001'
,'CrossCore Failed Exception Score 9013'
,'CrossCore Mitek Id Authentication Failed'
,'Fail CIP Checks Crosscore Solicited'
,'Fail PreciseID Checks Crossscore'
,'Sentilink Check Failed'
,'Failed Negative List Check'
,'Declined: Failed Hotfile Check'
,'Crosscore Verification Failed'
; 

/*DM campaign code*/
proc sql; 
	select distinct 
		quote(strip(substr(OFFER_CODE_FIRST_7,1,7)))
	into 
		:list_camp_codes separated by "," 
	from 
		&ds._mailfiles
	where CAMPFLAG1  in ('V','A','D','M','N','R','W','C')
	; 
quit; 

/* RM campaign code*/
proc sql; 
	select distinct 
		quote(strip(substr(OFFER_CODE_FIRST_7,1,7)))
	into 
		:list_re_camp_codes separated by "," 
	from 
		&ds._mailfiles
	where CAMPFLAG1  in ('B','G','I','J','Q');
	; 
quit;



proc sort data= &ds._mailfiles nodupkey out = &ds._mailfiles_dm1; by ReservationNumber; run; 


proc sql; 
	create table &ds._dm as 
	select 
		a. APPLICATION_ID, 
		b. DOB,
		c. RESERVATION_NUMBER, 
		substr(c. RESERVATION_NUMBER, 1, 7) as CampaignCode, 
		datepart(a. APPLICATION_RECEIVE_DATE) as AppRcvDate format date9., 
		datepart(a. READY_TO_BOARD_DATE) as RTBDate format date9., 
		datepart(a. BOARDDATE) as BoardDate format date9., 
		c. STATUS_DESC, 
		c. STATUS_NAME,
		case when c. STATUS_DESC in (&frauddecs.) then 1 else 0 end as FraudDecline_dm, 
		case when c. STATUS_DESC ^= 'Declined: Expired Offer' then 1 else 0 end as GrossResponse_dm, 
		case when a. BOARDDATE > "&startdate."d then 1 else 0 end as NetResponse_dm
	from 
		CAPSRPT.V_APPLICATION_CI a
		left join 
		CAPSRPT.V_APPLICANT_CI b
		   on a.APPLICATION_ID = b.APPLICATION_ID
		left join CAPSRPT.V_APPLICATION_STATUS c
			on a. APPLICATION_ID = c. APPLICATION_ID 
	where 
		"&maxappdate."d>= a.APPLICATION_RECEIVE_DATE >= "&minappdate."d /* date constraint avoids possible duplicate old Resv# */ 
		and (calculated CampaignCode in (&list_camp_codes.) or calculated CampaignCode in (&list_re_camp_codes.))
	order by 
		c. RESERVATION_NUMBER, a. APPLICATION_RECEIVE_DATE
; 
quit; 

proc sort data = &ds._dm nodupkey out = &ds._dm1; by RESERVATION_NUMBER; run; /* de-dup applications to retain earliest */ 

/*rm applicants*/
/*proc sql; 
	create table &ds._rm as 
	select 
		a. ExperianPin, 
		a. RequestNumber, 
		a. DateOfBirth as DOB,
		datepart(a. ApplicationReceivedDate) as AppRcvDate_rm format date9., 
		datepart(a. ReadyToBoardDate) as RTBDate_rm format date9., 
		datepart(a. BoardDate) as BoardDate_rm format date9., 
		b. StatusDescription, 
		case when b. StatusDescription in (&frauddecs.) then 1 else 0 end as FraudDecline_rm, 
		case when b. StatusDescription ^= 'Declined: Expired Offer' then 1 else 0 end as GrossResponse_rm, 
		case when a. BoardDate > "&startdate."d then 1 else 0 end as NetResponse_rm
	from 
		newdw.FactApplication a, 
		newdw.dimApplicationStatus b
	where 
		substr(a. RequestNumber, 1, 7) in (&list_re_camp_codes.)
		and"&maxappdate."d >=a.ApplicationReceivedDate >= "&minappdate."d
		and a. dimapplicationstatusid = b. dimapplicationstatusid
	order by 
		a. RequestNumber, a. ApplicationReceivedDate 
; 
quit; 
proc sort data = &ds._rm nodupkey out = &ds._rm1; by RequestNumber; run; /* de-dup applications to retain earliest */ 

/*pq applicants*/
/*proc sql; 
	create table &ds._pq as 
select  
   a.APPLICATION_ID,  
   a.APPLICATION_RECEIVE_DATE as AppRcvDate,
   b.APPLICANT_ID,  
   b.FIRST_NAME,  
   b.LAST_NAME,  
   b.ADDRESS1,  
   b.ADDRESS2,  
   scan(b. ADDRESS1, 1, ' ') as StreetNumber,
   b.DOB, 
   b.SSN_LAST_4,  
   c.PROGRAM_CODE,  
   c.RESERVATION_NUMBER,  
   c.STATUS_DESC,  
   c.STATUS_NAME  
from  
   CAPSRPT.V_APPLICATION_CI a  
   left join CAPSRPT.V_APPLICANT_CI b  
       on a.APPLICATION_ID = b.APPLICATION_ID  
   left join CAPSRPT.V_APPLICATION_STATUS c  
       on a.APPLICATION_ID = c.APPLICATION_ID  
where  
   a.APPLICATION_RECEIVE_DATE between "&minappdate."d and "&maxappdate."d
   and c.PROGRAM_CODE = "PQ01"  
order by  
   b.FIRST_NAME, b.LAST_NAME,  StreetNumber, a.APPLICATION_RECEIVE_DATE; 
quit; 

proc sort data = &ds._pq nodupkey out = &ds._pq1; by FIRST_NAME LAST_NAME StreetNumber; run;*/ /* de-dup applications via engineered match key to retain earliest */ 

%mend getresponse_trm;


/*************************Macro 1.3 - Step3. combined response, apply final response logic *******************/
%macro finalresponse_trm(startdate,ds); 

libname UnixFile "/sasdata/unix/Risk_Credit/Data_Strategy/ProgramCode_Dashboard/Data";

proc sql; 
	create table &ds._responseds as 
	select
		a. *, 
		b. AppRcvDate, 
		b. BoardDate, 
		b. FraudDecline_dm as FraudDecline, 
		b. GrossResponse_dm as GrossResponse,
		b. NetResponse_dm as NetResponse,
		c. finalcombinedaf as annual_fee
		/*c. AppRcvDate_rm, 
		c. BoardDate_rm, 
		c. FraudDecline_rm, 
		c. GrossResponse_rm,
		c. NetResponse_rm,
		d. APPLICATION_ID as PQappID,
		case when b. dob is not null then b. dob
		when c.dob is not null then c.dob
		else d.dob end as dateofbirth*/
	from &ds._mailfiles_dm1 a 
		left join &ds._dm1 b 
			on a. ReservationNumber = b. RESERVATION_NUMBER 
		left join unixfile.v_programcode c
			on substr(a.offer_code_first_7,1,4) = c.ProgramCode 
		/*left join &ds._rm1 c 
			on a. ENCRYPTED_PIN = c. ExperianPin
		left join &ds._pq1 d 
			on a. SURNAME = d. LAST_NAME
			and a. FIRST_NAME = d. FIRST_NAME  
			and a. STREET_NUMBER = d.StreetNumber*/
; 
quit; 

data &ds._finalresponse; /*update - keep all grr=2*/
	set &ds._responseds; 

	/*if PQappID ^= . then PQappMatch = 1; /* indicator for overlapping PQ app */ 
	/*else PQappMatch = 0; 

	/*if frauddecline_dm = 1 | frauddecline_rm = 1 then FraudDecline = 1;  /* tagged as fraud if either DM or Remail app is fraud declined */ 
	/*else FraudDecline = 0; 
	
	if grossresponse_dm = 1 | grossresponse_rm = 1 then GrossResponse = 1; /* tagged as 1 if either DM or Remail app is present and not declined for Expired Offer */ 
/*	else GrossResponse = 0; 

	if netresponse_dm = 1 | netresponse_rm = 1 then NetResponse = 1; /* tagged as 1 if either DM or Remail app has a Board Date */ 
	/*else NetResponse = 0; 

	if FraudDecline = 1 then GrossResponse = 2; /* all fraud declines are indeterminate */ 

    /*if pqappmatch = 1 and grossresponse_dm ^= 1 and grossresponse_rm ^= 1 then GrossResponse = 2; /* PQ apps without a DM app are indeterminate */ 
	
	if vantage3<640  and prospect_type="Prospecting" then scorecard =1;
	if vantage3>=640  and prospect_type="Prospecting" then scorecard =2;
	if vantage3<640  and prospect_type="Retargeting" then scorecard =3;
	if vantage3>=640  and prospect_type="Retargeting" then scorecard =4;

	if vantage3 >= 530 and vantage3 <=549 then vs_band="530-549";
	else if vantage3 >=550 and vantage3 <=600 then vs_band="550-600";
	else if  vantage3 >=601 and vantage3 <=639 then vs_band="601-639";
	else if  vantage3 >=640 and vantage3 <=700 then vs_band="640-700";
	else if  vantage3 >=701 and vantage3 <=730 then vs_band="701-730";
	else if  vantage3 >= 731 then vs_band="731-830" ;


	if campflag1 in ('F', 'V', 'E') then prospectingflag = 1; else prospectingflag = 0; 
	if campflag1 in ('A', 'W') then pqabandon_flag = 1; else pqabandon_flag = 0;
	if campflag1 in ('C','G') then prchargeoff_flag = 1; else prchargeoff_flag = 0; 
	if campflag1 in ('N', 'R','B','J') then prclosure_flag = 1; else prclosure_flag = 0; 
	if campflag1 in ('D', 'M','Q') then prdecline_flag = 1; else prdecline_flag = 0; 
run; 




%mend finalresponse_trm;

%macro assign_psi_tier;
if TRM10_Score >=0.02908089 and TRM10_Score <=1 then total_decile=1;
else if TRM10_Score >=0.0178773 and TRM10_Score <=0.02908088 then total_decile=2;
else if TRM10_Score >=0.01394068 and TRM10_Score <=0.01787729 then total_decile=3;
else if TRM10_Score >=0.01147906 and TRM10_Score <=0.01394067 then total_decile=4;
else if TRM10_Score >=0.00966261 and TRM10_Score <=0.01147905 then total_decile=5;
else if TRM10_Score >=0.00832388 and TRM10_Score <=0.0096626 then total_decile=6;
else if TRM10_Score >=0.00726601 and TRM10_Score <=0.00832387 then total_decile=7;
else if TRM10_Score >=0.00636179 and TRM10_Score <=0.007266 then total_decile=8;
else if TRM10_Score >=0.00522037 and TRM10_Score <=0.00636178 then total_decile=9;
else if TRM10_Score >=0 and TRM10_Score <=0.00522036 then total_decile=10;

if TRM10_Score >=0.02656043 and TRM10_Score <=1 then sc1_decile=1;
else if TRM10_Score >=0.01589291 and TRM10_Score <=0.02656042 then sc1_decile=2;
else if TRM10_Score >=0.0123958 and TRM10_Score <=0.0158929 then sc1_decile=3;
else if TRM10_Score >=0.01030504 and TRM10_Score <=0.01239579 then sc1_decile=4;
else if TRM10_Score >=0.0087901 and TRM10_Score <=0.01030503 then sc1_decile=5;
else if TRM10_Score >=0.00778552 and TRM10_Score <=0.00879009 then sc1_decile=6;
else if TRM10_Score >=0.00706834 and TRM10_Score <=0.00778551 then sc1_decile=7;
else if TRM10_Score >=0.00651691 and TRM10_Score <=0.00706833 then sc1_decile=8;
else if TRM10_Score >=0.00606842 and TRM10_Score <=0.0065169 then sc1_decile=9;
else if TRM10_Score >=0 and TRM10_Score <=0.00606841 then sc1_decile=10;

if TRM10_Score >=0.01668371 and TRM10_Score <=1 then sc2_decile=1;
else if TRM10_Score >=0.01153198 and TRM10_Score <=0.0166837 then sc2_decile=2;
else if TRM10_Score >=0.00931244 and TRM10_Score <=0.01153197 then sc2_decile=3;
else if TRM10_Score >=0.00798018 and TRM10_Score <=0.00931243 then sc2_decile=4;
else if TRM10_Score >=0.00703765 and TRM10_Score <=0.00798017 then sc2_decile=5;
else if TRM10_Score >=0.00635339 and TRM10_Score <=0.00703764 then sc2_decile=6;
else if TRM10_Score >=0.00569877 and TRM10_Score <=0.00635338 then sc2_decile=7;
else if TRM10_Score >=0.00513585 and TRM10_Score <=0.00569876 then sc2_decile=8;
else if TRM10_Score >=0.00421288 and TRM10_Score <=0.00513584 then sc2_decile=9;
else if TRM10_Score >=0 and TRM10_Score <=0.00421287 then sc2_decile=10;

if TRM10_Score >=0.0536694 and TRM10_Score <=1 then sc3_decile=1;
else if TRM10_Score >=0.02849683 and TRM10_Score <=0.05366939 then sc3_decile=2;
else if TRM10_Score >=0.02063868 and TRM10_Score <=0.02849682 then sc3_decile=3;
else if TRM10_Score >=0.01707402 and TRM10_Score <=0.02063867 then sc3_decile=4;
else if TRM10_Score >=0.01475292 and TRM10_Score <=0.01707401 then sc3_decile=5;
else if TRM10_Score >=0.01279835 and TRM10_Score <=0.01475291 then sc3_decile=6;
else if TRM10_Score >=0.01113234 and TRM10_Score <=0.01279834 then sc3_decile=7;
else if TRM10_Score >=0.00960001 and TRM10_Score <=0.01113233 then sc3_decile=8;
else if TRM10_Score >=0.00805475 and TRM10_Score <=0.0096 then sc3_decile=9;
else if TRM10_Score >=0 and TRM10_Score <=0.00805474 then sc3_decile=10;

if TRM10_Score >=0.02486282 and TRM10_Score <=1 then sc4_decile=1;
else if TRM10_Score >=0.01759291 and TRM10_Score <=0.02486281 then sc4_decile=2;
else if TRM10_Score >=0.01432997 and TRM10_Score <=0.0175929 then sc4_decile=3;
else if TRM10_Score >=0.01228554 and TRM10_Score <=0.01432996 then sc4_decile=4;
else if TRM10_Score >=0.01070027 and TRM10_Score <=0.01228553 then sc4_decile=5;
else if TRM10_Score >=0.00958126 and TRM10_Score <=0.01070026 then sc4_decile=6;
else if TRM10_Score >=0.00867152 and TRM10_Score <=0.00958125 then sc4_decile=7;
else if TRM10_Score >=0.00785271 and TRM10_Score <=0.00867151 then sc4_decile=8;
else if TRM10_Score >=0.00629435 and TRM10_Score <=0.0078527 then sc4_decile=9;
else if TRM10_Score >=0 and TRM10_Score <=0.00629434 then sc4_decile=10;
%mend;

%macro rollup(startdate,ds); 
proc sql;
	create table &ds._rollup as 
		select Prospect_type, 
			pqabandon_flag, 
			prchargeoff_flag,  
			prclosure_flag, 
			prdecline_flag, 
			vs_band, 
			annual_fee, 
			times_mailed_12mo_cnt, 
			trm10_tier,
			scorecard,
			rm_flag,
			count(*) as volume, 
			sum(GrossResponse) as responders, 
			sum(TRM_Score) as expected_responses,
			sum(EXP_RESPONSE_SCORE) as expected_responses_xpm,
			calculated responders/calculated volume as GRR,
			sum(NetResponse) as Boards, 
			calculated Boards/calculated volume as NRR
		from trm.&ds._finalresponse
			group by 
				Prospect_type, 
				pqabandon_flag, 
				prchargeoff_flag,  
				prclosure_flag, 
				prdecline_flag, 
				vs_band, annual_fee, 
				times_mailed_12mo_cnt, 
				trm10_tier,
				scorecard,
				rm_flag
			order by 
				Prospect_type, 
				pqabandon_flag, 
				prchargeoff_flag,  
				prclosure_flag, 
				prdecline_flag, 
				vs_band, annual_fee, 
				times_mailed_12mo_cnt, 
				trm10_tier,
				scorecard,
				rm_flag;
quit;


%mend;

/*
=======================================================================================================
 										Section 5: Variable Stability Macro
=======================================================================================================
*/


/******************************** Macro 5.1 - Pull Premiers *************************************************/

%macro append_prescreen_premier(input,output,name);
proc sql; 
create table &output._premiers as
     select a.*,
b.* from &input. a left join (select EPIN
           , PREMIER_V1_2_ALL7516     , PREMIER_V1_2_ALL7120     , PREMIER_V1_2_ALL6980  , PREMIER_V1_2_ALL1361
           , PREMIER_v1_2_IQT9416     , PREMIER_v1_2_IQT9413     , PREMIER_v1_2_IQT9415  , PREMIER_v1_2_IQT9410
           , PREMIER_V1_2_BCC5030     , PREMIER_V1_2_BCC5830     , PREMIER_V1_2_ALL7110  , PREMIER_V1_2_BCC5020
           , PREMIER_V1_2_ALL0300     , PREMIER_V1_2_ALL0336     , PREMIER_V1_2_ALL0400  , PREMIER_V1_2_ALL0416     
           , PREMIER_V1_2_ALL1380     , PREMIER_V1_2_ALL2002     , PREMIER_V1_2_ALL2005  , PREMIER_V1_2_ALL2350
           , PREMIER_V1_2_ALL2390     , PREMIER_V1_2_ALL2488     , PREMIER_V1_2_ALL2717D , PREMIER_V1_2_ALL5012
           , PREMIER_V1_2_ALL5070     , PREMIER_V1_2_ALL5320     , PREMIER_V1_2_ALL5361  , PREMIER_V1_2_ALL6200
           , PREMIER_V1_2_ALL6220     , PREMIER_V1_2_ALL6230     , PREMIER_V1_2_ALL6250  , PREMIER_V1_2_ALL6400
           , PREMIER_V1_2_ALL6900     , PREMIER_V1_2_ALL7371D    , PREMIER_V1_2_ALL7460  , PREMIER_V1_2_ALL8020
           , PREMIER_V1_2_ALL8110     , PREMIER_V1_2_ALL8120     , PREMIER_V1_2_ALL8121  , PREMIER_V1_2_ALL8221
           , PREMIER_V1_2_ALL8320     , PREMIER_V1_2_ALL8321     , PREMIER_V1_2_ALL8370  , PREMIER_V1_2_ALL8552
           , PREMIER_V1_2_ALL8560     , PREMIER_V1_2_ALL9110     , PREMIER_V1_2_ALL9210  , PREMIER_V1_2_AUA2320
           , PREMIER_V1_2_AUA8220     , PREMIER_V1_2_AUA8320     , PREMIER_V1_2_AUT5620  , PREMIER_V1_2_BCA6210
           , PREMIER_V1_2_BCA8151     , PREMIER_V1_2_BCC2328     , PREMIER_V1_2_BCC2388  , PREMIER_V1_2_BCC2800
           , PREMIER_V1_2_BCC3456     , PREMIER_V1_2_BCC3512     , PREMIER_V1_2_BCC5120  , PREMIER_V1_2_BCC5320
           , PREMIER_V1_2_BCC5400     , PREMIER_V1_2_BCC5421     , PREMIER_V1_2_BCC5422  , PREMIER_V1_2_BCC7117
           , PREMIER_V1_2_BCC7147     , PREMIER_V1_2_BCC7516     , PREMIER_V1_2_BCC7800  , PREMIER_V1_2_BCX0438
           , PREMIER_V1_2_BRC5620     , PREMIER_V1_2_BRC6280     , PREMIER_V1_2_COL3203  , PREMIER_V1_2_COL3211
           , PREMIER_V1_2_COL3237     , PREMIER_V1_2_COL5060     , PREMIER_V1_2_COL8192  , PREMIER_V1_2_ILN0438
           , PREMIER_V1_2_ILN1380     , PREMIER_V1_2_ILN2380     , PREMIER_V1_2_ILN5020  , PREMIER_V1_2_ILN6160
           , PREMIER_V1_2_IQB9416     , PREMIER_V1_2_IQB9510     , PREMIER_V1_2_IQF9510  , PREMIER_V1_2_IQT9417
           , PREMIER_V1_2_IQT9510     , PREMIER_V1_2_MTA2320     , PREMIER_V1_2_MTA2388  , PREMIER_V1_2_MTA6230
           , PREMIER_V1_2_PIL0438     , PREMIER_V1_2_PIL6200     , PREMIER_V1_2_REH5020  , PREMIER_V1_2_REH5320
           , PREMIER_V1_2_REV0300     , PREMIER_V1_2_REV2388     , PREMIER_V1_2_REV3421  , PREMIER_V1_2_REV3424
           , PREMIER_V1_2_REV3511     , PREMIER_V1_2_REV5036     , PREMIER_V1_2_REV5742  , PREMIER_V1_2_REV6220
           , PREMIER_V1_2_REV7470     , PREMIER_V1_2_REV7801     , PREMIER_V1_2_REV8220  , PREMIER_V1_2_RTA8151
           , PREMIER_V1_2_RTR1380     , PREMIER_V1_2_RTR3422     , PREMIER_V1_2_RTR3511  , PREMIER_V1_2_RTR6200
           , PREMIER_V1_2_RTR8320     , PREMIER_V1_2_STU2558     , PREMIER_V1_2_STU8320  , PREMIER_V1_2_UTI4180			
from datalake.ExperianPrescreenPremier where FileName like &name.) b
  on a.Encrypted_Pin = b.EPIN; 
run; 
%mend append_prescreen_premier;

/******************************** Macro 5.2 - Feature Engineering *************************************************/

%macro feature_engg;

/*Effetive_max_util*/

	/*checks for sustained utilization rate(uses max bal)*/
	if PREMIER_V1_2_ALL7120 > 990 then ALL7120tr = 230; /*Exceptions*/
	else ALL7120tr = min(PREMIER_V1_2_ALL7120, 200);

	if PREMIER_V1_2_ALL8370 > 9990 then ALL8370tr = 230; /*Exceptions*/
	else ALL8370tr = min(PREMIER_V1_2_ALL8370, 200);

	Effetive_max_util = (ALL7120tr * ALL8370tr)/100;

/*Full_delinq*/

	/*number of times more than 60day delinquent is zoomed up with max delinquent ever amplified the bad behavior*/
	ALL2350tr = min(PREMIER_V1_2_ALL2350 + 10, 28);

	if PREMIER_V1_2_ALL6980 > 990 then ALL6980tr = 15;
	else ALL6980tr = min(PREMIER_V1_2_ALL6980 + 10, 1000);

	Full_delinq_min = min(ALL2350tr * ALL6980tr, 10000);
	if Full_delinq_min <= 500 then Full_delinq = roundz(Full_delinq_min, 50);
	else if Full_delinq_min <= 1000 then Full_delinq = roundz(Full_delinq_min, 100);
	else Full_delinq = roundz(Full_delinq_min, 750);

/*Maxbal_mon_pymt_rate*/

	/*Average number of payments to be made w.r.t maximum balance*/

	if PREMIER_V1_2_BCC5120 > 999999990 then BCC5120tr = 25000; /*Exceptions*/
	else if PREMIER_V1_2_BCC5120 <= 10 then BCC5120tr = 9;
	else if PREMIER_V1_2_BCC5120 <= 100 then BCC5120tr = ROUNDZ(PREMIER_V1_2_BCC5120, 10);
	else if PREMIER_V1_2_BCC5120 <= 1000 then BCC5120tr = ROUNDZ(PREMIER_V1_2_BCC5120, 100);
	else if PREMIER_V1_2_BCC5120 <= 10000 then BCC5120tr = ROUNDZ(PREMIER_V1_2_BCC5120, 500);
	else BCC5120tr = ROUNDZ(min(PREMIER_V1_2_BCC5120, 20000), 1000);


	if PREMIER_V1_2_BCC5830 > 999999990 then BCC5830tr = 2500; /*Exceptions*/
	else if PREMIER_V1_2_BCC5830 <= 10 then BCC5830tr = 9;
	else if PREMIER_V1_2_BCC5830 <= 100 then BCC5830tr = ROUNDZ(PREMIER_V1_2_BCC5830, 10);
	else BCC5830tr = ROUNDZ(min(PREMIER_V1_2_BCC5830, 2000), 100);

	Maxbal_mon_pymt_rate = min((BCC5830tr / BCC5120tr) * 100, 60);

/*Totbal_mon_pymt_rate*/
	/*Average number of payments to be made w.r.t Total balance*/

	if PREMIER_V1_2_BCC5020 > 999999990 then BCC5020tr = 25000; /*Exceptions*/
	else if PREMIER_V1_2_BCC5020 <= 10 then BCC5020tr = 9;
	else if PREMIER_V1_2_BCC5020 <= 100 then BCC5020tr = ROUNDZ(PREMIER_V1_2_BCC5020, 10);
	else if PREMIER_V1_2_BCC5020 <= 1000 then BCC5020tr = ROUNDZ(PREMIER_V1_2_BCC5020, 100);
	else if PREMIER_V1_2_BCC5020 <= 10000 then BCC5020tr = ROUNDZ(PREMIER_V1_2_BCC5020, 500);
	else BCC5020tr = ROUNDZ(min(PREMIER_V1_2_BCC5020, 20000), 1000);


	if PREMIER_V1_2_BCC5830 > 999999990 then BCC5830tr = 2500; /*Exceptions*/
	else if PREMIER_V1_2_BCC5830 <= 10 then BCC5830tr = 9;
	else if PREMIER_V1_2_BCC5830 <= 100 then BCC5830tr = ROUNDZ(PREMIER_V1_2_BCC5830, 10);
	else BCC5830tr = ROUNDZ(min(PREMIER_V1_2_BCC5830, 2000), 100);

	Totbal_mon_pymt_rate = min((BCC5830tr / BCC5020tr) * 100, 60);

/*Util_NewCards_rate*/

	/*utilization per recently opened card(6 months)*/
		if PREMIER_V1_2_ALL7110 > 990 then ALL7110tr = 240; /*Exceptions*/
		else ALL7110tr = min(PREMIER_V1_2_ALL7110+10, 210);

		if PREMIER_V1_2_ALL7516 > 990 then ALL7516tr = 140; /*Exceptions*/
		else ALL7516tr = min(PREMIER_V1_2_ALL7516 + 10, 130);

		Util_NewCards_rate = (ALL7516tr / ALL7110tr) * 10;

/*Variables created by Matt*/

/*iqbt_16ratio*/

	iqbt_16ratio = premier_v1_2_iqb9416/premier_v1_2_iqt9416; 
	if premier_v1_2_iqt9416 = 0 then iqbt_16ratio = -1; 

/*iqt_1315ratio*/

	/*30days vs 3 mos*/
	iqt_1315ratio = premier_v1_2_iqt9413/premier_v1_2_iqt9415; 
	if premier_v1_2_iqt9415 = 0 then iqt_1315ratio = -1; 

/*iqt_1317grad*/

	/*30days vs 12 mos*/
	iqt_1317ratio = premier_v1_2_iqt9413/premier_v1_2_iqt9417; 
	if premier_v1_2_iqt9417 = 0 then iqt_1317ratio = -1; 


	iqt_1317grad = 2*premier_v1_2_iqt9413 - premier_v1_2_iqt9417;

/*iqt_1510grad*/

	/*3mo vs all time*/
	iqt_1510ratio = premier_v1_2_iqt9415/premier_v1_2_iqt9410; 
	if premier_v1_2_iqt9410 = 0 then iqt_1510ratio = -1; 

	iqt_1510grad = 2*premier_v1_2_iqt9415 - premier_v1_2_iqt9410; 

%mend feature_engg;


%macro trm_sc2trans;

/*bcc5421_woe*/
	/*PREMIER_V1_2_BCC5421 - Credit amount on most recent open */ 
	if PREMIER_V1_2_BCC5421 > 999999990 then bcc5421_impcap = -1; 
	else if PREMIER_V1_2_BCC5421 = 0 then bcc5421_impcap = 0; 
	else bcc5421_impcap = max(1000, min(round(PREMIER_V1_2_BCC5421, 1000),  5000))/1000; 

	if bcc5421_impcap = -1 then bcc5421_woe =  -0.221847752 ;
 	else if bcc5421_impcap = 1 then bcc5421_woe =  -0.615782366 ;
	else if bcc5421_impcap = 2 then bcc5421_woe =  0.049639017 ;
	else if bcc5421_impcap = 3 then bcc5421_woe =  0.0846879412 ;
	else if bcc5421_impcap = 4 then bcc5421_woe =  0.408789384 ;
	else bcc5421_woe =  0.5849793963 ;


/*	iqt9510 - months since recent inquiry*/
	if premier_v1_2_iqt9510 > 9990 then iqt9510_impcap = 8; 
	else iqt9510_impcap = min(premier_v1_2_iqt9510, 11) ; 

	iqt9510_impcaplog = log(1 + iqt9510_impcap); 


/*PREMIER_V1_2_REV3511 - Total number of revolving trades opened in the last 6 months with a balance > $0 excluding derogatory trades */	
	rev3511_cap = min(premier_v1_2_rev3511, 2); 
	if PREMIER_V1_2_REV3511 > 90 then rev3511_cap = 3; 

%mend trm_sc2trans;



%let additionalvar = 
	AppRcvDate	AppRcvDate_rm	BoardDate	BoardDate_rm
	CAMPAIGN_CODE	CAMPAIGN_DATE	CAMPFLAG1	CAMPFLAG2
	CRS18_P	CRS18_SEGMENT	DOB	DOB_format	ENCRYPTED_PIN	ENDMARK
	EPIN	FraudDecline	FraudDecline_dm	FraudDecline_rm
	GrossResponse	GrossResponse_dm	GrossResponse_rm
	NRM16	NRM16_TIER	NRM16_TIER_VS4	NetResponse	NetResponse_dm
	NetResponse_rm	OFFER_CODE_FIRST_7	OFFER_CODE_LAST_3
	PQappID	PQappMatch	PROSPECT_TYPE	RESERVATION_SEQUENCE
	RESPONSE_MODEL_TYPE	RISK_SCORE	RISK_SCORE_TYPE
	ReservationNumber	SCORE_VANTAGE_VALUE_V3_SEGMENT
	TIMES_MAILED_12MO_CNT
	TM12_3BR
	TM12_3RR
	TRM_TIER 
	TRM10_TIER
	TRM_Score	dateofbirth	scorecard	vantage3 age;

%let sc1_var = 
	vantage3  TM12_3RR PREMIER_V1_2_ALL8120
	PREMIER_V1_2_ALL1380  PREMIER_V1_2_ALL6250  PREMIER_V1_2_ALL8020
	PREMIER_V1_2_ALL8110  PREMIER_V1_2_ALL5070  PREMIER_V1_2_BCC5400
	PREMIER_V1_2_AUA2320  PREMIER_V1_2_AUT5620  PREMIER_V1_2_BCC7147
	PREMIER_V1_2_IQB9510  PREMIER_V1_2_COL5060  PREMIER_V1_2_REV7801
	PREMIER_V1_2_REV8220  PREMIER_V1_2_REV5036  PREMIER_V1_2_RTR1380
	PREMIER_V1_2_ALL5012  PREMIER_V1_2_BCA8151  PREMIER_V1_2_BCC3512
	PREMIER_V1_2_IQB9416  PREMIER_V1_2_MTA6230  PREMIER_V1_2_ALL7460
	PREMIER_V1_2_BCC5421  PREMIER_V1_2_ILN5020  PREMIER_V1_2_IQT9417
	PREMIER_V1_2_REV0300  PREMIER_V1_2_RTR6200  PREMIER_V1_2_ALL7371D
	PREMIER_V1_2_REV5742  iqt_1317ratio  iqbt_16ratio  Full_delinq
	ALL8370tr  Totbal_mon_pymt_rate  ALL7516tr  Util_NewCards_rate
;

%let sc2_var = 
		vantage3  TM12_3RR  PREMIER_V1_2_ALL5320
       PREMIER_V1_2_ALL8320  PREMIER_V1_2_ALL6250  PREMIER_V1_2_ALL0400
       PREMIER_V1_2_ALL8121  PREMIER_V1_2_BCC5400  PREMIER_V1_2_BCX0438
       PREMIER_V1_2_AUA8220  PREMIER_V1_2_ALL9210  PREMIER_V1_2_ALL2005
       PREMIER_V1_2_ILN0438  PREMIER_V1_2_ALL2002  PREMIER_V1_2_IQB9510
       PREMIER_V1_2_ILN1380  PREMIER_V1_2_RTR8320  PREMIER_V1_2_REV5036
       PREMIER_V1_2_RTA8151  PREMIER_V1_2_MTA2320  PREMIER_V1_2_STU8320
       PREMIER_V1_2_REV7470  PREMIER_V1_2_BCA6210  PREMIER_V1_2_REV6220
       PREMIER_V1_2_BCC5422  PREMIER_V1_2_BRC5620  PREMIER_V1_2_BCC7800
       PREMIER_V1_2_BRC6280  PREMIER_V1_2_ALL2488  PREMIER_V1_2_IQB9416
       PREMIER_V1_2_ALL6400  PREMIER_V1_2_ALL5361  PREMIER_V1_2_ALL7460
       PREMIER_V1_2_ALL8560  PREMIER_V1_2_ALL2350  PREMIER_V1_2_ALL8321
       iqt9510_impcaplog  bcc5421_woe  rev3511_cap  Effetive_max_util
       Maxbal_mon_pymt_rate    

;

%let sc3_var = 
		PREMIER_V1_2_BCC7516	PREMIER_V1_2_MTA2388	ALL2350tr	Full_delinq
		PREMIER_V1_2_REV3511	PREMIER_V1_2_ALL5320	PREMIER_V1_2_REV3421
		PREMIER_V1_2_ALL0336	Util_NewCards_rate	PREMIER_V1_2_ALL9110	PREMIER_V1_2_STU2558
		PREMIER_V1_2_BCC2800	PREMIER_V1_2_ALL7460	PREMIER_V1_2_COL3203	PREMIER_V1_2_REH5320
		iqt_1510grad	PREMIER_V1_2_IQT9510	ALL8370tr	PREMIER_V1_2_ALL8560
		PREMIER_V1_2_AUA2320	PREMIER_V1_2_BCA8151	PREMIER_V1_2_ALL6200
		PREMIER_V1_2_COL8192	PREMIER_V1_2_ALL2717D	iqt_1317grad	PREMIER_V1_2_ALL6230
		PREMIER_V1_2_ALL0300	PREMIER_V1_2_ALL2488	PREMIER_V1_2_ALL8221
		PREMIER_V1_2_IQF9510	PREMIER_V1_2_ALL1361	PREMIER_V1_2_BCC7117	PREMIER_V1_2_BCC2388
		PREMIER_V1_2_REV2388	iqt_1315ratio	PREMIER_V1_2_PIL6200
		PREMIER_V1_2_RTR3422	iqbt_16ratio	PREMIER_V1_2_ALL0400	PREMIER_V1_2_BCC3456
		PREMIER_V1_2_ILN2380	TM12_3RR	prdecline_flag	prclosure_flag	pqabandon_flag	prchargeoff_flag
		scorecard  TRM_Score OFFER_CODE_FIRST_7 campflag1

;


%LET sc4_var = 
	Effetive_max_util	Full_delinq	iqt_1510grad	iqbt_16ratio	vantage3	prclosure_flag	prdecline_flag	prchargeoff_flag
	PREMIER_V1_2_ALL0300	PREMIER_V1_2_ALL0336	PREMIER_V1_2_ALL0416	PREMIER_V1_2_ALL2390
	PREMIER_V1_2_ALL2488	PREMIER_V1_2_ALL5320	PREMIER_V1_2_ALL6220	PREMIER_V1_2_ALL7460	PREMIER_V1_2_ALL8221	PREMIER_V1_2_ALL8370
	PREMIER_V1_2_ALL8552	PREMIER_V1_2_ALL8560	PREMIER_V1_2_ALL9110	PREMIER_V1_2_AUA8320
	PREMIER_V1_2_BCC2328	PREMIER_V1_2_BCC5120	PREMIER_V1_2_BCC5320	PREMIER_V1_2_BCC5400
	PREMIER_V1_2_BCC7117	PREMIER_V1_2_COL3211	PREMIER_V1_2_IQT9510	PREMIER_V1_2_PIL0438	PREMIER_V1_2_REH5020	PREMIER_V1_2_REV2388
	PREMIER_V1_2_REV3424	PREMIER_V1_2_STU2558	TM12_3RR
;

%macro run_one_month(reportdate);
    %local starttime monthdate reportmon startdate labelname reptname;
    %let starttime = %sysfunc(datetime());
    %put NOTE: Start time = %sysfunc(putn(&starttime, datetime19.));
    /* reportdate should be like 01JAN2025 */
    %let monthdate = %sysfunc(intnx(month, "&reportdate"d, 0, b));
    %let reportmon = %sysfunc(putn(&monthdate, monname3.));
    %let startdate = %sysfunc(putn(&monthdate, date9.));
    %let labelname = %sysfunc(putn(&monthdate, monyy5.));
    %let reptname  = %sysfunc(putn(&monthdate, monyy7.));
    %put NOTE: Running month=&labelname startdate=&startdate reportmon=&reportmon reptname=&reptname;
    options validvarname=any;
    %readmailfile_trm(&startdate., &labelname.);
    %getresponse_trm(&startdate., &labelname.);
    %finalresponse_trm(&startdate., &labelname.);
    %rollup(&startdate., &labelname.);
    data trm.&labelname._finalresponse;
        set &labelname._finalresponse;
        %assign_psi_tier;
    run;
    proc export data=&labelname._rollup
        outfile="&folder./exp_&labelname._rollup.csv"
        dbms=csv
        replace;
    run;
%mend;
%macro run_months(start=01JAN2025, end=01APR2026);
    %local i n monthdate reportdate;
    %let n = %sysfunc(intck(month, "&start"d, "&end"d));
    %do i = 0 %to &n;
        %let monthdate = %sysfunc(intnx(month, "&start"d, &i, b));
        %let reportdate = %sysfunc(putn(&monthdate, date9.));
        %run_one_month(&reportdate.);
    %end;
%mend;
%run_months(start=01JAN2025, end=01APR2026);

data combined;
set jan25_rollup feb25_rollup mar25_rollup apr25_rollup may25_rollup jun25_rollup jul25_rollup aug25_rollup sep25_rollup oct25_rollup nov25_rollup dec25_rollup jan26_rollup feb26_rollup mar26_rollup apr26_rollup indsname=dsname;

length MonthIndicator $10;
MonthIndicator = dsname; /* This takes the libname.tablename */
run;

proc export data=combined
    outfile="&folder./combined_rollup_3.csv"
    dbms=csv
    replace;
run;



'''
)

{'LOG': "\x1439                                                         The SAS System                               11:34 Thursday, June 4, 2026\n\n318        ods listing close;ods html5 (id=saspy_internal) file=_tomods1 options(bitmap_mode='inline') device=svg style=HTMLBlue;\n318      ! ods graphics on / outputfmt=png;\nNOTE: Writing HTML5(SASPY_INTERNAL) Body file: _TOMODS1\n319        \n320        \n321        /*CLI Boarded Accounts Pulling*/\n322        \n323        libname Daily1 '/sasdata/unix/Risk_Credit/Data_Strategy/Portfolio_Daily_CLI_Dashboard/Daily_Vintage' access=readonly;\nNOTE: Libref DAILY1 was successfully assigned as follows: \n      Engine:        V9 \n      Physical Name: /sasdata/unix/Risk_Credit/Data_Strategy/Portfolio_Daily_CLI_Dashboard/Daily_Vintage\n324        \n325        \n326        data cli;\n327        \n328        Set Daily1.dailyvintage_2025: Daily1.dailyvintage_2026:;\n329        \n330        /*where clitype = 'Early 1';*/\n331        \n332        ru

In [45]:
cnx.SAS_Connection.list_tables()

[('CLI', 'DATA'), ('CLI_BOARDEDACCOUNT_JAN25TOJUN26', 'DATA')]

In [46]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd


In [47]:
df_saspull_total=cnx.SAS_Connection.sd2df(table='CLI_BOARDEDACCOUNT_JAN25TOJUN26',libref='WORK')

In [48]:
table2 = pa.Table.from_pandas(df_saspull_total, preserve_index=False)
# Write the Arrow Table to a Parquet file
pq.write_table(table2, 'df_saspull_total.parquet')

In [49]:
cnx.close_connections

<bound method C1BConnections.close_connections of <Agora.C1BConnections object at 0x0000018ECDA97740>>

In [51]:
df_saspull_total.shape

(12574654, 2)